# **Black Friday Purchase Prediction**
**Balanced Approach: Ridge Regression + Random Forest Regressor**

### Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import FunctionTransformer

In [ ]:
# 1) Load data
file_path = "../data/raw/black_friday.xlsx"
df = pd.read_excel(file_path)

df.head()

In [ ]:
# 2) Basic cleanup
df.columns = [c.strip() for c in df.columns]

print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())

In [ ]:
# 3) Define target and features
# Purchase is the numeric target to predict
target = "Purchase"

# Remove rows with missing target if any
df = df.dropna(subset=[target])

# To avoid leakage or overly specific memorization
# User_ID and Product_ID are identifiers; you can exclude them for a cleaner,
# more interpretable business model
drop_cols = [target, "User_ID", "Product_ID"]

X = df.drop(columns=drop_cols)
y = df[target]

print("\nFeature columns used:\n", X.columns.tolist())

In [ ]:
# 4) Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
# 5) Preprocessing
# Numeric: impute missing values, then scale for Ridge
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical: impute missing values, then one-hot encode
#categorical_transformer = Pipeline(steps=[
#    ("imputer", SimpleImputer(strategy="most_frequent")),
#    ("onehot", OneHotEncoder(handle_unknown="ignore"))
#])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant',
                              fill_value='Missing')),
    ('to_string',
     FunctionTransformer(lambda x: x.astype(str))),
    ('onehot',
     OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", cat_pipeline, categorical_features)
    ]
)

In [ ]:
# 6) Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("\nTraining set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

In [ ]:
# 7) Models
ridge_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10.0))
])

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
# 8) Fit models
ridge_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

In [ ]:
# 9) Predictions
ridge_pred = ridge_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

In [ ]:
# 10) Evaluation function
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print("-" * len(name))
    print(f"MAE : {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2  : {r2:.4f}")

In [ ]:
# 11) Evaluate both
evaluate_model("Ridge Regression", y_test, ridge_pred)
evaluate_model("Random Forest Regressor", y_test, rf_pred)

In [ ]:
# 12) Interpretability: Ridge coefficients
# Get feature names after preprocessing
feature_names = ridge_model.named_steps["preprocessor"].get_feature_names_out()
ridge_coefficients = ridge_model.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": ridge_coefficients,
    "AbsCoefficient": np.abs(ridge_coefficients)
}).sort_values(by="AbsCoefficient", ascending=False)

print("\nTop 20 Ridge features by absolute coefficient:")
print(coef_df[["Feature", "Coefficient"]].head(20))

In [ ]:
# 13) Interpretability: Random Forest permutation importance
perm = permutation_importance(
    rf_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "Feature": X.columns,
    "ImportanceMean": perm.importances_mean,
    "ImportanceStd": perm.importances_std
}).sort_values(by="ImportanceMean", ascending=False)

print("\nRandom Forest permutation importance:")
print(perm_df.head(20))

In [ ]:
# 14) Save outputs for reporting
coef_df.to_csv("ridge_coefficients.csv", index=False)
perm_df.to_csv("rf_permutation_importance.csv", index=False)

In [ ]:
# 15) Recommendation logic
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print("\nModel recommendation:")
if rf_rmse < ridge_rmse:
    print("Use Random Forest as the primary model for prediction, and Ridge for explanation.")
else:
    print("Use Ridge as the primary model since it balances accuracy and interpretability well.")